# NER and Text Classification

**Goal:** Extract structured entities from text and classify text into categories.

**Prereqs:** Complete [00_tokenization_and_embeddings.ipynb](./00_tokenization_and_embeddings.ipynb) first.

**Libraries:** `spacy`, `transformers`

**Setup:** Run `python -m spacy download en_core_web_sm` if you haven't already.

## Part 1: Named Entity Recognition (NER)

**Go/TS comparison:** In a web service, you parse JSON into structs with known fields. NER does something similar for natural language — it finds structured entities (people, places, organizations, dates, money) hiding in unstructured text. Think of it as `json.Unmarshal()` for English.

In [ ]:
import spacy

# Load the small English model
nlp = spacy.load("en_core_web_sm")

# Process a sentence — NER happens automatically
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")

print("Entities found:")
for ent in doc.ents:
    print(f"  {ent.text:20s}  {ent.label_:10s}  ({spacy.explain(ent.label_)})")

In [ ]:
# Try different types of text
texts = [
    "Elon Musk announced that Tesla will open a new factory in Austin, Texas by March 2025.",
    "The European Central Bank raised interest rates by 0.25% on Thursday.",
    "Dr. Sarah Chen published her research on quantum computing at MIT.",
]

for text in texts:
    doc = nlp(text)
    print(f"Text: {text[:60]}...")
    for ent in doc.ents:
        print(f"  {ent.text:25s} → {ent.label_}")
    print()

In [ ]:
# Experiment: NER isn't perfect — see where it fails
tricky_texts = [
    "I love Paris Hilton",                    # Paris = person or place?
    "Apple stock rose after the iPhone launch", # Apple = company, not fruit
    "Jordan played basketball in Jordan",      # Same word, different entities
    "Reading is a city in England",            # Reading = place, not the verb
]

for text in tricky_texts:
    doc = nlp(text)
    print(f"'{text}'")
    if doc.ents:
        for ent in doc.ents:
            print(f"  {ent.text} → {ent.label_}")
    else:
        print("  (no entities found)")
    print()

### ✍️ In Your Own Words

Why does NER sometimes get things wrong? What makes entity recognition ambiguous? Write your answer here.

### Entity Frequency Analysis

In [ ]:
# Process a longer text and count entity types
text = """
Google announced a partnership with NASA to advance quantum computing research.
CEO Sundar Pichai met with NASA Administrator Bill Nelson in Washington, D.C.
on January 15, 2025. The deal is worth approximately $500 million over five years.
Microsoft and Amazon are also investing heavily in quantum technology.
The European Union has allocated €1 billion for quantum research through 2030.
"""

doc = nlp(text)

# Count entity types
from collections import Counter
entity_counts = Counter(ent.label_ for ent in doc.ents)

print("Entity frequency:")
for label, count in entity_counts.most_common():
    print(f"  {label:10s} ({spacy.explain(label):30s}): {count}")

print(f"\nAll entities:")
for ent in doc.ents:
    print(f"  {ent.text:25s} → {ent.label_}")

### ✍️ In Your Own Words

How could entity frequency analysis be useful in a real data pipeline? Write your answer here.

## Part 2: Text Classification

**Go/TS comparison:** In a web service, you route requests to handlers based on the URL path or method. Text classification does the same for natural language — it assigns a label to text so you can route it, filter it, or act on it. Sentiment analysis is the most common example.

### Sentiment Analysis

In [ ]:
from transformers import pipeline

# Load a pretrained sentiment analysis model
classifier = pipeline("sentiment-analysis")

sentences = [
    "I absolutely love this product, it's amazing!",
    "This is the worst experience I've ever had.",
    "The movie was okay, nothing special.",
    "I'm not unhappy with the results.",           # Double negative
    "Great product, terrible customer service.",     # Mixed
]

for sentence in sentences:
    result = classifier(sentence)[0]
    print(f"[{result['label']:8s} {result['score']:.3f}]  {sentence}")

In [ ]:
# Experiment: sarcasm and nuance — where models struggle
sarcastic = [
    "Oh great, another Monday morning.",
    "Wow, what a surprise, the train is late again.",
    "Sure, because that worked so well last time.",
]

print("Sarcasm test (models usually miss this):")
for s in sarcastic:
    result = classifier(s)[0]
    print(f"  [{result['label']:8s} {result['score']:.3f}]  {s}")

### ✍️ In Your Own Words

Why do models struggle with sarcasm? What would it take to detect sarcasm reliably? Write your answer here.

### Zero-Shot Classification

**Go/TS comparison:** This is the powerful one. Imagine a Go HTTP router where you could add new routes at runtime without writing new handler code. Zero-shot classification lets you define categories *at runtime* — no training needed. The model uses natural language inference to score how well the text matches each label.

In [ ]:
# Zero-shot: you provide the labels at runtime
zero_shot = pipeline("zero-shot-classification")

text = "The new iPhone has an amazing camera but the battery life is disappointing"

result = zero_shot(
    text,
    candidate_labels=["technology", "sports", "food", "politics"],
)

print(f"Text: {text}\n")
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label:15s}  {score:.3f}")

In [ ]:
# Experiment: change the labels and watch scores shift
texts_and_labels = [
    (
        "The stock market crashed after the Fed raised interest rates",
        ["finance", "politics", "technology", "sports"],
    ),
    (
        "The patient was diagnosed with type 2 diabetes",
        ["medical", "fitness", "nutrition", "insurance"],
    ),
    (
        "The team scored a last-minute goal to win the championship",
        ["sports", "business", "entertainment", "politics"],
    ),
]

for text, labels in texts_and_labels:
    result = zero_shot(text, candidate_labels=labels)
    print(f"Text: {text[:60]}...")
    for label, score in zip(result['labels'][:3], result['scores'][:3]):
        print(f"  {label:15s}  {score:.3f}")
    print()

In [ ]:
# Experiment: labels that are close in meaning
text = "I just bought a new gaming laptop"
label_sets = [
    ["tech", "technology", "gadgets", "electronics"],    # Synonyms
    ["shopping", "gaming", "computers", "hardware"],     # Different aspects
]

for labels in label_sets:
    result = zero_shot(text, candidate_labels=labels)
    print(f"Labels: {labels}")
    for label, score in zip(result['labels'], result['scores']):
        print(f"  {label:15s}  {score:.3f}")
    print()

### ✍️ In Your Own Words

How does zero-shot classification work under the hood? Why can it classify text into categories it was never trained on? Write your answer here.

### Confidence Thresholds

In [ ]:
# Not all predictions are confident — filter by threshold
texts = [
    "Apple released a new MacBook Pro with M3 chip",
    "I went for a walk in the park this morning",
    "The government announced new tax policies for 2025",
    "My cat knocked over the Christmas tree again",
    "SpaceX successfully launched 50 Starlink satellites",
]

labels = ["technology", "politics", "lifestyle", "science"]
threshold = 0.5

print(f"Threshold: {threshold}\n")
for text in texts:
    result = zero_shot(text, candidate_labels=labels)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    status = "✓" if top_score >= threshold else "✗ (low confidence)"
    print(f"  [{top_score:.3f}] {top_label:12s} {status}  — {text[:50]}")

### ✍️ In Your Own Words

When would you use a confidence threshold in production? What's the trade-off between a high and low threshold? Write your answer here.

## Recap

- ✅ **NER** extracts structured entities (PERSON, ORG, GPE, DATE, MONEY) from text
- ✅ spaCy provides production-grade NER out of the box
- ✅ NER is statistical, not perfect — ambiguity is inherent in language
- ✅ **Sentiment analysis** classifies text as positive/negative with confidence scores
- ✅ Models struggle with sarcasm, negation, and mixed sentiment
- ✅ **Zero-shot classification** defines categories at runtime — no training needed
- ✅ It works via natural language inference (NLI) — treating each label as a hypothesis
- ✅ Confidence thresholds filter uncertain predictions — trade precision for recall

**You've completed the NLP Fundamentals section!** These concepts (tokenization, embeddings, similarity, NER, classification) are the building blocks used in the RAG app.